# MQ heater warm-up curves

How long after power-on the MQ-2 and MQ-135 read steadily in clean air. The answer
sets `CONFIG_FIREX_MQ_WARMUP_S` (the console refuses a baseline before it) and, in M2,
the fusion warm-up that blocks every alarm after boot.

**Procedure.** Leave the node unpowered for at least an hour. Power it, start
`python tools/capture/capture.py --port COMx --out captures/cold-start-<date> --period 2000`
straight away, and keep the room air clean and still for 30 minutes. Repeat on three
different days; new sensors also need their 24-48 h burn-in first, which is a separate
first-power-on run.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # run from tools/calibration, next to fxcal.py
import fxcal
import matplotlib.pyplot as plt

In [ ]:
CAPTURE = None  # e.g. "../../captures/cold-start-2026-10-01.csv"

cap = fxcal.load_capture(CAPTURE) if CAPTURE else fxcal.demo_warmup()
if not CAPTURE:
    print("SYNTHETIC DATA. Set CAPTURE to a bench recording before drawing any conclusion.")

In [ ]:
TOLERANCE = 0.02  # settled once Rs stays within 2 % ...
HOLD_S = 60       # ... for a full minute

fig, ax = plt.subplots(figsize=(9, 4))
for column in ("mq2_rs", "mq135_rs"):
    ts, vs = fxcal.series(cap, column)
    if not ts:
        print(f"{column}: no data")
        continue
    settle = fxcal.warmup_time(ts, vs, TOLERANCE, HOLD_S)
    line, = ax.plot([t / 60 for t in ts], vs, label=column)
    if settle is None:
        print(f"{column}: did not settle within this capture")
    else:
        ax.axvline(settle / 60, linestyle="--", color=line.get_color())
        print(f"{column}: settled {settle:.0f} s after power-on")
ax.set_xlabel("minutes since power-on")
ax.set_ylabel("Rs (ohm)")
ax.set_yscale("log")
ax.legend()
plt.show()

Record the slower of the two settle times, rounded up, with the date and the sensor
serials. Use the worst of the three days, not the average.